In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

In [56]:
def get_parameter_count(model): 
    all_params =       sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return all_params, trainable_params

In [59]:
def get_model(name, num_classes=3):
    match name:
        case "swin_t":
            model = models.swin_t(
                weights = models.Swin_T_Weights.IMAGENET1K_V1,
            )
            model.head = nn.Linear(model.head.in_features, num_classes)
            return model
        
        case "swin_b":
            model = models.swin_b(
                weights = models.Swin_B_Weights.IMAGENET1K_V1,
            )
            model.head = nn.Linear(model.head.in_features, num_classes)
            return model

        case "visionTransformer":
            model = models.vit_b_16(
                weights = models.ViT_B_16_Weights.IMAGENET1K_V1,
            )
            fc: nn.Linear = model.heads.head # type: ignore[assignment]
            model.heads.head = nn.Linear(fc.in_features, num_classes)
            return model
    
    return None

In [61]:
model_name = (
    "swin_t"
    # "swin_b"
    # "visionTransformer"
)
print('loading model:', model_name)

torch.manual_seed(0)
num_classes = 3
model = get_model(model_name, num_classes)
if model is None:
    raise NameError(f"No such model yet: {model_name}")
save_name = f"../pretrained_backbone/ckpt_{model_name}.pt"
torch.save(model.state_dict(), save_name)

loading model: swin_t


In [64]:
lr_final = 0.01
epochs = 12
lr0 = 1e-3
lr_delta = (lr0 - lr0 * lr_final) / (epochs-1)
print(lr_delta)
for epoch in range(epochs):
    lr = lr0 - lr_delta * epoch
    print(f'lr: {lr:5f}')

9e-05
lr: 0.001000
lr: 0.000910
lr: 0.000820
lr: 0.000730
lr: 0.000640
lr: 0.000550
lr: 0.000460
lr: 0.000370
lr: 0.000280
lr: 0.000190
lr: 0.000100
lr: 0.000010
